# Build a tool-calling agent in Python (SDK)

You will build a **data analyst**. You ask it a question in plain English — *"Which product
generated the most revenue?"* — and it writes SQL, runs that SQL against a SQLite file on this
machine, and explains the rows that came back.

The model decides *whether* to query and *what* to query. Your code only runs the query it asks
for. That split is the whole idea of a tool.

| Piece | What it does | Who runs it |
|---|---|---|
| `store.db` | a small SQLite file this notebook creates: 8 products, 15 orders | your code |
| `query_database` | the tool in the catalog: a name, a description, one `sql` argument | the platform stores it |
| `run_query_database` | the Python that actually opens the database and runs the SELECT | **your code** |
| `sql-analyst-agent` | the prompt: system message, the table schema, the bound tool, the default model | the platform stores it |
| `run_prompt_with_tools` | the loop: model turn, tool call, model turn again, until there is an answer | the SDK |

Every cell runs against a real account. Nothing here is faked or mocked.

**Two ways to do every step.** Each step that creates something has two headings:
**In the dashboard**, with the values to type and a screenshot, and **The same thing in code**,
with a cell to run. They are not two different features — the dashboard and the SDK call the same
API, so the result is identical. Pick either. Doing both is harmless, because every code cell
looks for what already exists before it creates anything.

**Three kinds of code cell.** Most of this notebook is not the thing you would ship. Every cell's
lead-in says which kind it is:

| Label | What it is | Goes in your app? |
|---|---|---|
| **Setup** | creates something on the platform, once. The dashboard does the same job. | no |
| **Your app** | the code that would really ship | **yes** |
| **Check** | proves the step worked, or shows what just happened | no |
| **Broken on purpose** | a failure being demonstrated | no |

**Companion page:** [Build a tool-calling agent in Python (SDK)](https://docs.acruxcore.com/docs/tutorials/build-a-tool-calling-agent-in-python-sdk)

---

## Step 0 — What you need before you start

Four things:

1. **An AcruxCore account** and a **personal API key**. In the dashboard, open
   **Account & keys → New key**, name it `sql-agent`, and copy the value the moment it appears.
   That is the only time the full key is shown.
2. **A model connected to the gateway**, so the prompt has something to run on. This notebook
   uses the public name `claude-haiku`, on a direct Anthropic credential. Any model works —
   the public name is yours to choose, and your code never learns which provider answered.
3. **Python 3.9 or newer.**
4. **Nothing else.** The database, the tool, the prompt and the binding are all created below.

#### If you have no model yet

A **model** is a public name your code sends as `"model"`, mapped to an upstream model on one of
your provider credentials. Open **Gateway → Models → New model** and fill in:

| Field | What to enter |
|---|---|
| **Public name** | `claude-haiku` — this is the name your code and your prompt will use |
| **Credential** | your provider credential, for example Anthropic |
| **Upstream model** | the provider's own id, for example `claude-haiku-4-5-20251001` |
| **Prices** | leave blank; they auto-fill for known models |

![New model dialog with public name claude-haiku, an Anthropic credential, and upstream model claude-haiku-4-5-20251001](https://docs.acruxcore.com/img/tutorials/build-a-tool-calling-agent-in-python-sdk/02-new-model.png)

Click **Register model**, then **Test** on the new row to fire a one-token completion and confirm
the key works. Any model works here — if you use a different public name, change `MODEL` below.

In [ ]:
%pip install -q --upgrade acruxcore

**Setup.** Set your key and the base URL, and name the things this notebook will create.

A key typed into a notebook is saved *inside the notebook file*. Prefer setting these in your
shell before you start Jupyter, and treat this cell as a fallback.

In [1]:
import json
import os

# Better: export these in your shell before starting Jupyter.
os.environ.setdefault("ACRUXCORE_API_KEY", "acx_sk_...")
os.environ.setdefault("ACRUXCORE_BASE_URL", "https://api.acruxcore.com/api/v1")

MODEL = "claude-haiku"           # a public name from Gateway -> Models; any model works
TOOL = "query_database"          # the tool this notebook creates
PROMPT = "sql-analyst-agent"     # the prompt this notebook creates
DB_PATH = "store.db"             # the SQLite file, next to this notebook

# Do NOT print the base URL: the saved output would publish whatever host you ran against.

### Preflight

**Check.** Run this before anything else. It checks things in the order they usually fail, so a
missing key gives you one clear line instead of a stack trace from deep inside a later call.

In [2]:
import inspect

import httpx

import acruxcore
from acruxcore import AcruxCore
from acruxcore.gateway_api import GatewayNamespace

hub = AcruxCore()          # reads ACRUXCORE_API_KEY / ACRUXCORE_BASE_URL

# 1. Does the key work at all?
await hub.prompts.list(limit=1)
print("api key: ok")

# 2. Does this SDK version know client_tools? Step 8 needs it.
has_client_tools = "client_tools" in inspect.signature(
    GatewayNamespace.run_prompt_with_tools
).parameters
print("client_tools supported:", has_client_tools)
if not has_client_tools:
    print(f"  !! acruxcore {acruxcore.__version__} is too old - upgrade it")

# 3. Is there a model to run on, and is MODEL one of them?
#    Models have no SDK namespace yet, so call the endpoint directly.
async with httpx.AsyncClient() as http:
    res = await http.get(
        f"{os.environ['ACRUXCORE_BASE_URL']}/gateway/models",
        headers={"Authorization": f"Bearer {os.environ['ACRUXCORE_API_KEY']}"},
    )
models = [m["publicName"] for m in res.json()]
print("models on this team:", models or "NONE - add one in Gateway -> Models")
print(f"MODEL {MODEL!r} available:", MODEL in models)

api key: ok
client_tools supported: True
models on this team: ['mistral-small', 'llama-3.3-70b', 'claude-haiku', 'gemini-flash', 'gpt-4o-mini']
MODEL 'claude-haiku' available: True


---

## Step 1 — Who runs the tool

### The general problem

A tool has two halves, and they do not have to live in the same place.

- The **definition** is what the model reads: a name, a description, and a schema of arguments.
  It is text. The model needs it to decide whether to call the tool and what to pass.
- The **implementation** is the code that does the work. The model never sees it.

Every platform that offers tools has to decide who owns each half. Some keep both in your
process. Some keep both on their servers.

### Where our case sits

AcruxCore stores the definition in the **tool catalog**, and lets you choose where the
implementation lives. That choice is the version's **executor**:

| Executor | Who runs the implementation | Good for |
|---|---|---|
| **HTTP** | the gateway. You describe a request once; the platform makes it. | a public or reachable API |
| **Client** | *your* process. The platform stores the schema and deliberately no body. | anything only your machine can reach |

A SQLite file on your laptop is the textbook case for **client**. The gateway cannot open
`store.db` — there is no URL for it, and no network path to your disk. Your Python can. So the
catalog holds the `sql` schema, and your code holds the `sqlite3` call.

### What that means for the code you write

Because the body never leaves your process, the loop has to come back to you mid-run. That is
what `run_prompt_with_tools` does: it calls the model, and when the model asks for
`query_database`, it calls **your** function, feeds the result back, and asks the model again.

You supply the body as a map — `client_tools={"query_database": run_query_database}` — keyed by
the tool's catalog name. Nothing is written back to the catalog.

### The recommendation

Author a client tool in the dashboard (or with the API calls below), and pass its body from your
app. Reach for the `@acrux.tool` decorator instead when you want the function signature to be
the single source of the schema — Step 10 shows it, and
[Create a tool](https://docs.acruxcore.com/docs/guides/create-a-tool)
compares the two properly.

---

## Step 2 — Build the local database

There is nothing to do in the dashboard for this step. `store.db` is a file on your machine, not
an object on the platform.

**Setup.** The rows are fixed and inlined here, so the answers later in this notebook match, and
so a reader who downloaded only the `.ipynb` can still run it. Re-running drops and rebuilds the
two tables.

In [3]:
import sqlite3

PRODUCTS = [
    # (id, name, category, price, stock)
    (1, "Aeron Chair", "Furniture", 1395.00, 12),
    (2, "Standing Desk", "Furniture", 640.00, 30),
    (3, "Mechanical Keyboard", "Electronics", 110.00, 220),
    (4, "4K Monitor", "Electronics", 480.00, 85),
    (5, "USB-C Hub", "Electronics", 45.00, 500),
    (6, "Desk Lamp", "Lighting", 65.00, 140),
    (7, "Noise-Cancelling Headphones", "Electronics", 320.00, 60),
    (8, "Ergonomic Mouse", "Electronics", 75.00, 300),
]

ORDERS = [
    # (id, product_id, quantity, order_date, customer)
    (1, 3, 4, "2026-05-02", "Acme Corp"), (2, 4, 2, "2026-05-05", "Globex"),
    (3, 1, 1, "2026-05-11", "Initech"), (4, 5, 20, "2026-05-14", "Acme Corp"),
    (5, 7, 3, "2026-05-19", "Umbrella"), (6, 3, 10, "2026-05-22", "Globex"),
    (7, 2, 5, "2026-06-01", "Initech"), (8, 4, 4, "2026-06-03", "Umbrella"),
    (9, 8, 12, "2026-06-08", "Acme Corp"), (10, 1, 2, "2026-06-12", "Globex"),
    (11, 7, 6, "2026-06-15", "Initech"), (12, 5, 40, "2026-06-18", "Umbrella"),
    (13, 6, 8, "2026-06-21", "Acme Corp"), (14, 3, 15, "2026-06-25", "Globex"),
    (15, 4, 1, "2026-06-29", "Initech"),
]

SCHEMA_SQL = """
CREATE TABLE products (
    id INTEGER PRIMARY KEY, name TEXT, category TEXT, price REAL, stock INTEGER
);
CREATE TABLE orders (
    id INTEGER PRIMARY KEY, product_id INTEGER REFERENCES products(id),
    quantity INTEGER, order_date TEXT, customer TEXT
);
"""

conn = sqlite3.connect(DB_PATH)
conn.executescript("DROP TABLE IF EXISTS orders; DROP TABLE IF EXISTS products;" + SCHEMA_SQL)
conn.executemany("INSERT INTO products VALUES (?, ?, ?, ?, ?)", PRODUCTS)
conn.executemany("INSERT INTO orders VALUES (?, ?, ?, ?, ?)", ORDERS)
conn.commit()
conn.close()
print(f"seeded {DB_PATH}: {len(PRODUCTS)} products, {len(ORDERS)} orders")

seeded store.db: 8 products, 15 orders


**Check.** The one answer worth knowing before the model gets involved. If the agent later says
something different from this, the agent is wrong — not the database.

In [4]:
conn = sqlite3.connect(DB_PATH)
top = conn.execute(
    """
    SELECT products.name, SUM(orders.quantity * products.price) AS revenue
    FROM orders JOIN products ON orders.product_id = products.id
    GROUP BY products.id ORDER BY revenue DESC LIMIT 1
    """
).fetchone()
june = conn.execute(
    "SELECT SUM(quantity) FROM orders WHERE order_date BETWEEN '2026-06-01' AND '2026-06-30'"
).fetchone()[0]
conn.close()

print(f"top product by revenue: {top[0]} at ${top[1]:,.2f}")
print(f"units ordered in June 2026: {june}")

top product by revenue: Aeron Chair at $4,185.00
units ordered in June 2026: 93


---

## Step 3 — Create the `query_database` tool

A tool in the catalog is **two objects**, not one. The **shell** owns the name and the
model-facing description. A **version** owns the argument schema and the executor. The
dashboard does not split this into two screens: **New tool** writes both at once, and Step 4
below only repeats the second half in code.

### In the dashboard

**Gateway → Tools → New tool.** One dialog holds everything the tool needs.

| Field | What to enter |
|---|---|
| **Name** | `query_database` |
| **Description** | `Run a single read-only SQL SELECT against the store database and return the matching rows.` |
| **Parameters** | one row: name `sql`, type `string`, **required**, described as `A single read-only SQLite SELECT statement.` |
| **Executor** | **Client — your own code runs it** |

![New tool dialog with the name query_database, a description about running a read-only SQL SELECT, a required sql string parameter, and the executor set to Client — your own code runs it](https://docs.acruxcore.com/img/tutorials/build-a-tool-calling-agent-in-python-sdk/03-new-tool.png)

Click **Create tool**. That writes the tool and its first version together, and the first
version automatically gets the `production` and `staging` aliases.

The description is the sentence the **model** reads when it decides whether to call this tool.
Write it for the model, not for your teammates.

### The same thing in code

**Setup.** In code the shell and the version stay two separate calls — Step 4 below makes
the second one. This cell only creates the shell: find-or-create, so a second run of this
notebook creates nothing and prints `already in the catalog`.

In [5]:
TOOL_DESCRIPTION = (
    "Run a single read-only SQL SELECT against the store database and return the "
    "matching rows."
)


async def find_tool_by_name(name: str):
    """The catalog row with exactly this name, or None.

    A notebook helper, NOT an SDK function. `hub.tools.list(search=...)` matches
    substrings, so the exact-name filter has to happen here.
    """
    found = await hub.tools.list(search=name, limit=100)
    return next((t for t in found.data if t.name == name), None)


tool = await find_tool_by_name(TOOL)
if tool is None:
    tool = await hub.tools.create(name=TOOL, description=TOOL_DESCRIPTION)
    print(f"created tool shell {TOOL}")
else:
    print(f"tool {TOOL} already in the catalog")

print("tool id:", tool.id)

created tool shell query_database
tool id: ad2b32ad-93ac-4e75-a208-9e30c31c1957


---

## Step 4 — Commit version 1, with a client executor

The dashboard already did this: the **New tool** dialog in Step 3 sent the schema and the
executor in the same request that created the shell. In code the two stay separate calls, and
this one decides **who runs the tool** — for a local database, the answer is your own process.

### The same thing in code

**Setup.** A version is immutable, which is why changing a schema means committing a new version
rather than editing this one. This cell commits only when the tool has no versions yet.

In [6]:
SQL_SCHEMA = {
    "type": "object",
    "properties": {
        "sql": {
            "type": "string",
            "description": "A single read-only SQLite SELECT statement.",
        }
    },
    "required": ["sql"],
}

versions = await hub.tools.list_versions(tool.id, limit=1)
if versions.total == 0:
    version = await hub.tools.commit_version(
        tool.id,
        parameters_schema=SQL_SCHEMA,
        executor={"type": "client"},   # the platform stores the schema and no body
        description=TOOL_DESCRIPTION,
    )
    aliases = [a.alias for a in version.aliases]
    print(f"committed v{version.version_number}, aliases now pointing here: {aliases}")
else:
    print(f"tool already has {versions.total} version(s) - nothing committed")

committed v1, aliases now pointing here: ['production', 'staging']


**Check.** What the model will actually read. `resolve` answers the only question that matters
here: is this tool callable, and under which executor?

In [7]:
resolved = (await hub.tools.resolve([{"name": TOOL, "alias": "production"}]))[0]

print("executor:", resolved.executor_type, "(client = your process runs it)")
print("version:", resolved.version_number)
print(json.dumps(resolved.function, indent=2))

executor: client (client = your process runs it)
version: 1
{
  "name": "query_database",
  "description": "Run a single read-only SQL SELECT against the store database and return the matching rows.",
  "parameters": {
    "type": "object",
    "required": [
      "sql"
    ],
    "properties": {
      "sql": {
        "type": "string",
        "description": "A single read-only SQLite SELECT statement."
      }
    }
  }
}


---

## Step 5 — Create the prompt

The prompt holds the **system instructions**, and those instructions include the table schema —
otherwise the model has to guess your column names. It also holds the **default model**.

The user's question is *not* stored here. Your code appends it at run time, which is why one
prompt answers any number of different questions.

### In the dashboard

**Prompts → New prompt**, then the **Editor** tab.

| Field | What to enter |
|---|---|
| **Name** | `sql-analyst-agent` |
| **Description** | `Text-to-SQL data analyst.` |
| **Default model** | `claude-haiku` |
| **System message** | the text in the next code cell, `SYSTEM_PROMPT` — it is too long to repeat here, and a second copy would drift |

![The sql-analyst-agent prompt Editor tab showing default model claude-haiku, a system message with the database schema, and production pointing at v1](https://docs.acruxcore.com/img/tutorials/build-a-tool-calling-agent-in-python-sdk/05-prompt-editor.png)

Click **Commit version**. The model is part of the version, so committing bakes it in, and
because it is the first commit `production` points at `v1` automatically.

### The same thing in code

**Setup.** Two separate checks on purpose. A prompt shell with zero versions is a real state — an
earlier run that died between the two calls leaves one — and "the name exists" is not "it has
content".

If you already have a prompt called `sql-analyst-agent` from earlier work, this cell keeps it and
commits nothing. That is only what you want if its system message is the one below — otherwise
rename `PROMPT` at the top of the notebook, or delete the old prompt first.

In [8]:
SYSTEM_PROMPT = """You are a data analyst for an online store. Answer questions about products
and sales by querying a SQLite database with the query_database tool. Never guess - always query.

Schema:
CREATE TABLE products (id INTEGER PRIMARY KEY, name TEXT, category TEXT, price REAL, stock INTEGER);
CREATE TABLE orders (id INTEGER PRIMARY KEY, product_id INTEGER REFERENCES products(id), quantity INTEGER, order_date TEXT, customer TEXT);

Write a single read-only SQLite SELECT, call query_database with it, then answer in one or two
sentences using only the rows it returns. Prices are in USD; revenue = quantity * price;
order_date is YYYY-MM-DD."""


async def find_prompt_by_name(name: str):
    """The prompt with exactly this name, or None. A notebook helper, NOT an SDK function."""
    found = await hub.prompts.list(search=name, limit=100)
    return next((p for p in found.data if p.name == name), None)


prompt = await find_prompt_by_name(PROMPT)
if prompt is None:
    prompt = await hub.prompts.create(name=PROMPT, description="Text-to-SQL data analyst.")
    print(f"created prompt shell {PROMPT}")
else:
    print(f"prompt {PROMPT} already exists")

if (await hub.prompts.list_versions(prompt.id, limit=1)).total == 0:
    pv = await hub.prompts.commit_version(
        prompt.id,
        messages=[{"role": "system", "content": SYSTEM_PROMPT}],
        model=MODEL,          # baked into the version - your code never hardcodes it
    )
    print(f"committed prompt v{pv.version_number} on model {pv.model}")
else:
    print("prompt already has a version - nothing committed")

created prompt shell sql-analyst-agent
committed prompt v1 on model claude-haiku


---

## Step 6 — Connect the tool to the prompt

Right now the tool and the prompt know nothing about each other. A **binding** joins them, so one
`render` call returns the system message *and* the tool schema together.

The binding stores an **alias**, not a version number. Point it at `production` and the prompt
follows whatever you promote to `production` later — no code change.

### In the dashboard

**Prompts → `sql-analyst-agent` → Tools tab → + Connect a tool from the catalog.**

| Field | What to enter |
|---|---|
| **Tool** | `query_database` |
| **Alias** | `production` |
| **Column** | **default** — every alias of the prompt inherits it |

![The prompt Tools tab showing query database connected](https://docs.acruxcore.com/img/tutorials/build-a-tool-calling-agent-in-python-sdk/06-prompt-tools.png)

It saves straight away; there is no separate commit for a binding.

### The same thing in code

**Setup.** `set_tool_binding` replaces the binding for that tool instead of adding a second one,
so running this twice leaves one row, not two.

In [9]:
binding = await hub.prompts.set_tool_binding(prompt.id, tool.id, tool_alias="production")
print(f"bound {binding.tool_name} @ {binding.tool_alias} -> v{binding.resolved_version_number}")

bindings = await hub.prompts.list_tool_bindings(prompt.id)
print("tools on this prompt:", [b.tool_name for b in bindings.default])

bound query_database @ production -> v1
tools on this prompt: ['query_database']


**Check.** One render call, and everything the model needs comes back together. This is the
payload your app will hand to the loop.

In [10]:
rendered = await hub.prompts.render(PROMPT, "production")

print("model:      ", rendered.model)
print("version:    ", f"v{rendered.version_number}")
print("messages:   ", [m["role"] for m in rendered.messages])
print("tools:      ", [t["function"]["name"] for t in rendered.tools])
for r in rendered.tool_resolutions:
    print(f"resolution:  {r.name} @ {r.alias} -> v{r.version_number} (defined in the {r.source})")

model:       claude-haiku
version:     v1
messages:    ['system']
tools:       ['query_database']
resolution:  query_database @ production -> v1 (defined in the default)


---

## Step 7 — Write the implementation

The catalog holds the schema and no body. This cell is the body.

**Your app.** This ships. Two things in it are not decoration:

- **`mode=ro`** opens the file read-only at the SQLite level. The *model* chose this SQL, so
  treat it as untrusted input. A stray or hostile `UPDATE` cannot change your data if the
  connection itself cannot write.
- **the single-`SELECT` guard** rejects anything else before it reaches SQLite, and rejects a
  second statement smuggled in after a semicolon.

Never point a tool like this at a writable production database.

In [11]:
def run_query_database(sql: str) -> list[dict]:
    """Run one read-only SELECT against the local store database.

    The parameter is named `sql` because that is the catalog schema's own field name -
    the loop calls this as run_query_database(sql=...), not with one args dict.
    """
    statement = sql.strip().rstrip(";").strip()
    if not statement.lower().startswith("select"):
        raise ValueError("Only read-only SELECT statements are allowed.")
    if ";" in statement:
        raise ValueError("Only a single statement is allowed.")

    print(f"  -> query_database: {' '.join(statement.split())}")
    conn = sqlite3.connect(f"file:{DB_PATH}?mode=ro", uri=True)
    conn.row_factory = sqlite3.Row
    try:
        return [dict(row) for row in conn.execute(statement).fetchall()]
    finally:
        conn.close()


#: Keyed by catalog tool name. The catalog holds the schema and deliberately no body,
#: so this map is the whole of what your app contributes.
CLIENT_TOOLS = {TOOL: run_query_database}

print("implementation ready for:", list(CLIENT_TOOLS))

implementation ready for: ['query_database']


---

## Step 8 — Run the agent

**Your app.** This is the part that really ships, and it is short. Read it in four moves:

1. `render` fetches the system message, the tool schema and the model.
2. Your question is appended as a `user` message.
3. `run_prompt_with_tools` runs the loop and calls back into `CLIENT_TOOLS` for every tool call.
4. `trace` names the run and drops it into a session, so related runs group together.

There is **no hardcoded model**. `rendered.model` is whatever the prompt version says. Change the
model in the dashboard and this code follows it on the next render.

In [12]:
async def ask(question: str) -> str:
    """Answer one question with the stored prompt, the bound tool, and your implementation."""
    rendered = await hub.prompts.render(PROMPT, "production")
    result = await hub.gateway.run_prompt_with_tools(
        rendered,
        messages=[*rendered.messages, {"role": "user", "content": question}],
        client_tools=CLIENT_TOOLS,
        trace={"name": PROMPT, "session_id": "sql-agent-demo"},
    )
    TRACE_IDS.append(result.trace_id)
    print(f"  (trace {result.trace_id}, {result.iterations} model turns)")
    return result.content


TRACE_IDS: list = []
QUESTIONS = [
    "Which product generated the most total revenue, and how much?",
    "How many total units were ordered in June 2026?",
]

answers = {}
for question in QUESTIONS:
    print(f"\nQ: {question}")
    answers[question] = await ask(question)
    print(f"A: {answers[question]}")


Q: Which product generated the most total revenue, and how much?
  -> query_database: SELECT p.name, SUM(o.quantity * p.price) as total_revenue FROM products p JOIN orders o ON p.id = o.product_id GROUP BY p.id, p.name ORDER BY total_revenue DESC LIMIT 1
  (trace 6ef29c3c-3886-4862-bbb7-0b7e9abca0f5, 2 model turns)
A: The Aeron Chair generated the most total revenue at $4,185.00.

Q: How many total units were ordered in June 2026?
  -> query_database: SELECT SUM(quantity) as total_units FROM orders WHERE order_date LIKE '2026-06%'
  (trace e9245b16-ca40-4db8-aa24-0000b6d9a8b3, 2 model turns)
A: In June 2026, a total of 93 units were ordered.


Read that from the bottom up. Your `run_query_database` printed the SQL it was handed, so you can
see the model wrote a `JOIN` and a `GROUP BY` on its own — nothing in your code writes SQL. Then
the model turned the returned rows into a sentence.

Both answers should match the two numbers Step 2 printed. The wording will differ from run to
run; the numbers should not.

---

## Step 9 — Read the trace back

**Check.** Read it from the API rather than trusting a screenshot. Each question is **one**
trace, because the loop threads the same trace id through every model turn.

The **shape** is what to check: an LLM span that asked for the tool, the `query_database` tool
span your process reported nested underneath it, and a final LLM span that wrote the answer. The
gateway records the LLM spans; the SDK adds the tool span from your process, which is why the tool
span carries no tokens — no model was called to run your SQL.

Token counts, costs and timings move on every run, because the model does not answer identically
twice. Only the shape is stable.

In [13]:
detail = await hub.traces.get(TRACE_IDS[0])


def walk(spans, depth=0):
    for span in spans:
        # An llm span is named by its timestamp, so the model is the useful label.
        label = span.model if span.kind == "llm" else span.name
        print(
            f"{'  ' * depth}- [{span.kind}] {label}  {span.status}  "
            f"tokens={span.total_tokens or 0}"
        )
        walk(span.children, depth + 1)


print(f"trace: {detail.trace.name}   session: {detail.trace.session_id}")
print(f"spans: {detail.trace.span_count}   total tokens: {detail.trace.total_tokens}")
walk(detail.spans)

trace: sql-analyst-agent   session: sql-agent-demo
spans: 3   total tokens: 1762
- [llm] claude-haiku-4-5-20251001  ok  tokens=854
  - [tool] query_database  ok  tokens=0
- [llm] claude-haiku-4-5-20251001  ok  tokens=908


The dashboard shows the same tree, and clicking the tool span shows the payloads the loop
captured — the SQL the model wrote as the input, the rows your code returned as the output.

![Trace named sql-analyst-agent with three spans, an LLM span, a query database tool span, and a final LLM span, all marked OK](https://docs.acruxcore.com/img/tutorials/build-a-tool-calling-agent-in-python-sdk/08-trace-tree.png)

![The expanded query database span showing an Input with the SELECT statement and an Output with a single row for Aeron Chair](https://docs.acruxcore.com/img/tutorials/build-a-tool-calling-agent-in-python-sdk/09-trace-span.png)

Payloads are stored only while payload capture is on for your team. It is on by default; turn it
off in **Observability → Settings** if you would rather not store request bodies.

Both runs also collected under the session id you passed, `sql-agent-demo`. A session is just a
caller-supplied string that groups related runs — one user's conversation, one job, one test run.

![The sql-agent-demo session showing two sql-analyst-agent traces, each with 3 spans and a token count](https://docs.acruxcore.com/img/tutorials/build-a-tool-calling-agent-in-python-sdk/10-session.png)

---

## Step 10 — The other way to own the definition

**Your app.** Everything above kept the definition in the catalog and passed the body from your
app. The `@acrux.tool` decorator flips the ownership: the **function** becomes the single source
of the name, the description and the schema, all derived from the signature and the docstring.

One flag matters here. `sync=True` — the default — writes the derived schema back to the catalog
as a new version and moves the `production` alias to it. That is fine when code owns the
definition, and destructive when the dashboard does: it supersedes what somebody typed there.
This cell passes `sync=False`, so the definition is sent to the model for this run only and the
catalog keeps the version you committed in Step 4.

Pick one owner per tool. Never both. See
[Create a tool](https://docs.acruxcore.com/docs/guides/create-a-tool)
for the full comparison.

In [14]:
from acruxcore import acrux


@acrux.tool
def query_database(sql: str) -> list:
    """Run a read-only SQL SELECT against the store database.

    Args:
        sql: A single read-only SQLite SELECT statement.
    """
    return run_query_database(sql)


rendered = await hub.prompts.render(PROMPT, "production")
result = await hub.gateway.run_prompt_with_tools(
    rendered,
    messages=[*rendered.messages, {"role": "user", "content": "How many 4K Monitors are in stock?"}],
    tools=[query_database],
    sync=False,          # do NOT write this schema over the dashboard's version
    trace={"name": PROMPT, "session_id": "sql-agent-demo"},
)
print("A:", result.content)

still = (await hub.tools.resolve([{"name": TOOL, "alias": "production"}]))[0]
print(f"catalog production alias still points at v{still.version_number}")

  -> query_database: SELECT stock FROM products WHERE name LIKE '%4K Monitor%'
A: There are 85 4K Monitors in stock.
catalog production alias still points at v1


---

## Step 11 — Stream a plain answer

**Your app.** When you want tokens as they are generated, use `hub.gateway.stream` and iterate
it.

Streaming yields text deltas and does **not** run tools — that is what the loop in Step 8 is for.
So reach for streaming when the answer is prose, and for the loop when the model needs data.

In [15]:
stream = await hub.gateway.stream(
    MODEL,
    [{"role": "user", "content": "In one sentence, what makes a good data analyst?"}],
)
async for chunk in stream:
    print(chunk.delta.get("content", "") or "", end="", flush=True)
print()

A good data analyst combines technical proficiency with curiosity and communication skills to transform raw data into actionable insights that drive informed decisions.


---

## Step 12 — Four ways to get this wrong

Every cell in this step is **broken on purpose**. None of it is app code. Each one fails the way
it really fails, so you recognise the message when it happens to you.

### Mistake 1 — the tool is bound, but your app forgot the body

**Broken on purpose.** A `client` tool is a promise your process will answer. Break the promise
and the loop has nothing to call.

In [16]:
rendered = await hub.prompts.render(PROMPT, "production")
try:
    await hub.gateway.run_prompt_with_tools(
        rendered,
        messages=[*rendered.messages, {"role": "user", "content": "How many products are there?"}],
        client_tools={},        # broken on purpose: no body for query_database
    )
    print("no error - unexpected")
except Exception as err:
    print(f"{type(err).__name__}: {err}")

AcruxCoreError: acruxcore: tool 'query_database' has a client executor, so something has to run it, but no implementation was supplied. Pass it in client_tools={'query_database': ...}, or pass dispatch=. client_tools held: [].


### Mistake 2 — the function cannot take the schema's arguments

**Broken on purpose.** The loop calls your function with the schema's own field names as keyword
arguments. Rename the parameter and Python refuses the call, long before any SQL runs.

In [17]:
def wrong_signature(query: str) -> list:
    """Broken on purpose: the schema's field is `sql`, not `query`."""
    return run_query_database(query)


try:
    wrong_signature(sql="SELECT 1")       # exactly how the loop would call it
except TypeError as err:
    print(f"TypeError: {err}")
    print("fix: name the parameter after the schema field, `sql`")

TypeError: wrong_signature() got an unexpected keyword argument 'sql'
fix: name the parameter after the schema field, `sql`


### Mistake 3 — a non-SELECT reaches the tool

**Broken on purpose.** The model writes the SQL, so your body is the last line of defence. Both
guards fire before SQLite is asked to do anything.

In [18]:
for bad in [
    "UPDATE products SET price = 0",
    "SELECT name FROM products; DROP TABLE orders",
]:
    try:
        run_query_database(bad)
        print(f"ran - BAD: {bad}")
    except ValueError as err:
        print(f"rejected: {bad!r}\n  -> {err}")

rejected: 'UPDATE products SET price = 0'
  -> Only read-only SELECT statements are allowed.
rejected: 'SELECT name FROM products; DROP TABLE orders'
  -> Only a single statement is allowed.


### Mistake 4 — no tool reaches the model, and nothing complains

**Broken on purpose.** This is the quiet one. Pass no tools and the model still answers — from
the system prompt alone, with no data behind it. There is no error, no warning, and nothing red in
the trace.

In [19]:
silent = await hub.gateway.chat(
    MODEL,
    [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": "Which product generated the most total revenue, and how much?"},
    ],
    trace={"name": "no-tools-on-purpose"},
)
print(silent.content)
print("\n(compare against Step 2: Aeron Chair, $4,185.00)")

I'll query the database to find which product generated the most total revenue.

```sql
SELECT p.name, SUM(o.quantity * p.price) as total_revenue
FROM orders o
JOIN products p ON o.product_id = p.id
GROUP BY p.id, p.name
ORDER BY total_revenue DESC
LIMIT 1
```

<query_database>
<query>
SELECT p.name, SUM(o.quantity * p.price) as total_revenue
FROM orders o
JOIN products p ON o.product_id = p.id
GROUP BY p.id, p.name
ORDER BY total_revenue DESC
LIMIT 1
</query>
</query_database>

Based on the results, the product that generated the most total revenue is **[Product Name]** with **$[Amount]** in total revenue. This represents the sum of all quantities sold multiplied by the product's price across all orders in the database.

(compare against Step 2: Aeron Chair, $4,185.00)


Read that answer carefully. With no tool to call, the model has three ways out and it picks one
at random: it writes the SQL as text and stops — this run went further and typed out a
tool-call-shaped block of its own, which no tool ever received; it refuses, because the system
message told it never to guess; or it invents a product and a number that look completely
plausible. Only the third one is dangerous, and none of the three is distinguishable by looking at
the answer.

That is why the trace matters. A run that really queried has a `query_database` tool span. A run
that did not has two LLM spans and nothing in between.

---

## Step 13 — Close the client

**Your app.** Traces are reported in the background so they never slow your request down. Closing
the client flushes whatever is still queued. In a script `async with AcruxCore() as hub:` does it
for you; a notebook has no block to leave, so do it by hand.

In [20]:
await hub.gateway.aclose()
print("flushed")

flushed


---

## What you built

A prompt and a tool that live on the platform, and one function on your machine that the model
can reach. The model chose when to query, wrote the SQL, and read the rows back. Your code never
wrote a query and never named a model.

### What of this actually ships

This much, and nothing else from this notebook:

```python
import sqlite3
from acruxcore import AcruxCore


def run_query_database(sql: str) -> list[dict]:
    statement = sql.strip().rstrip(";").strip()
    if not statement.lower().startswith("select") or ";" in statement:
        raise ValueError("Only a single read-only SELECT is allowed.")
    conn = sqlite3.connect("file:store.db?mode=ro", uri=True)
    conn.row_factory = sqlite3.Row
    try:
        return [dict(row) for row in conn.execute(statement).fetchall()]
    finally:
        conn.close()


async def ask(question: str) -> str:
    async with AcruxCore() as hub:
        rendered = await hub.prompts.render("sql-analyst-agent", "production")
        result = await hub.gateway.run_prompt_with_tools(
            rendered,
            messages=[*rendered.messages, {"role": "user", "content": question}],
            client_tools={"query_database": run_query_database},
            trace={"name": "sql-analyst-agent", "session_id": "sql-agent-demo"},
        )
        return result.content
```

Everything else was scaffolding:

- `find_tool_by_name` and `find_prompt_by_name` exist so this notebook can be re-run. They are
  notebook helpers, not SDK calls.
- the create-and-commit cells are the dashboard's job, done once.
- every **Check** cell — the preflight, `resolve`, `render`, the trace walk — proves a step
  worked. None of it belongs in a request path.
- Step 12 is all deliberately broken.

### What this notebook left in your team

- a tool `query_database` at v1, `production` and `staging` both pointing at it
- a prompt `sql-analyst-agent` at v1, with `claude-haiku` baked into the version
- a binding joining the two, inherited by every prompt alias
- a session `sql-agent-demo` holding every run above

### Where to go next

- [Build a tool-calling agent in Python (no SDK)](https://docs.acruxcore.com/docs/tutorials/build-a-tool-calling-agent-in-python-no-sdk)
  — the same loop written by hand over plain REST, so you can see what the SDK does for you.
- [Create a tool](https://docs.acruxcore.com/docs/guides/create-a-tool)
  — the decorator from Step 10, properly compared against the catalog route.
- [Using sessions and traces](https://docs.acruxcore.com/docs/guides/using-sessions-and-traces)
  — group related runs and dig into what happened.